# Experiment 025 — 30M Story→Math Developmental Shift

Formal T4×2 run designed for Kaggle **Save Version → Run All**. The whole job has an 8-hour wall-clock budget, including setup and finalization. Worker invocations checkpoint and automatically resume inside the same Kaggle job.


In [ ]:
import subprocess, sys
from pathlib import Path

ROOT = Path('/kaggle/working/mini-cells')
BRANCH = 'codex/experiment-025-story-math-growth'
REPO = 'https://github.com/ArcheLabs/mini-cells.git'

if not (ROOT / '.git').exists():
    subprocess.run(['git', 'clone', '--branch', BRANCH, '--single-branch', REPO, str(ROOT)], check=True)
else:
    subprocess.run(['git', 'fetch', 'origin', BRANCH], cwd=ROOT, check=True)
    subprocess.run(['git', 'checkout', BRANCH], cwd=ROOT, check=True)
    subprocess.run(['git', 'pull', '--ff-only', 'origin', BRANCH], cwd=ROOT, check=True)

HEAD = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=ROOT, text=True).strip()
TREE = subprocess.check_output(['git', 'rev-parse', 'HEAD^{tree}'], cwd=ROOT, text=True).strip()
DIRTY = subprocess.check_output(['git', 'status', '--porcelain', '--untracked-files=no'], cwd=ROOT, text=True).strip()
print({'HEAD': HEAD, 'tree': TREE, 'tracked_dirty': bool(DIRTY)})
assert not DIRTY


In [ ]:
subprocess.run([
    sys.executable, '-m', 'pytest',
    'tests/test_story_math_shift_30m.py',
    'tests/test_clm_progressive_growth.py',
    'tests/test_growth_router.py',
    '-q',
], cwd=ROOT, check=True)


In [ ]:
import torch
gpu_names = [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]
print({'gpu_count': torch.cuda.device_count(), 'gpus': gpu_names})
assert torch.cuda.device_count() >= 2, 'Formal Experiment 025 expects T4×2'


## One-shot formal run

This cell owns the full unattended job. Defaults: 8.0h global wall budget, 30-minute finalization reserve, 2.5h worker slices, automatic checkpoint/resume, and 50M Story→Math shift tokens.


In [ ]:
subprocess.run([
    sys.executable,
    'scripts/run_experiment_025_story_math_growth.py',
    '--total-wall-hours', '8',
    '--finalization-reserve-minutes', '30',
    '--round-wall-hours', '2.5',
], cwd=ROOT, check=True)


In [ ]:
import json
from IPython.display import display, Image

OUT = ROOT / 'results' / 'experiment-025-story-math-growth'
summary_path = OUT / 'worker-summary.json'
summary = json.loads(summary_path.read_text()) if summary_path.is_file() else {}
print(json.dumps(summary, indent=2))
decision_path = OUT / 'decision.json'
if decision_path.is_file():
    decision = json.loads(decision_path.read_text())
    print(json.dumps(decision, indent=2))
    for name in ['story-math-performance.png', 'growth-timeline.png']:
        path = OUT / name
        if path.is_file():
            display(Image(filename=str(path)))
else:
    print('Formal run ended partial inside the 8h budget; checkpoints and partial metrics were preserved.')


## Automatic publication

If and only if both arms completed and `decision.json` exists, this cell publishes the curated non-checkpoint outputs to `kaggle/experiment-025-story-math-growth-results` using the existing `GITHUB_TOKEN` Kaggle secret. A publication failure does not erase the completed local Kaggle outputs.


In [ ]:
if bool(summary.get('complete')) and decision_path.is_file():
    publish = subprocess.run([
        sys.executable,
        'scripts/publish_experiment_025_story_math_growth.py',
        '--push',
    ], cwd=ROOT, text=True, capture_output=True)
    print(publish.stdout)
    if publish.returncode != 0:
        print('Publication failed; completed Kaggle outputs remain available.')
        print(publish.stderr)
else:
    print('Publication skipped because the formal run is incomplete.')
